# NYSC Survival: generate Nigerian voices with YarnGPT (free), v2

This version speaks **one sentence at a time** with natural pauses between sentences, slows the delivery slightly, and **re-tries** any line that comes out too short or too long. It **replaces** all earlier voice files.

This notebook turns every line in `lines.json` into a Nigerian-accented MP3, using [YarnGPT](https://huggingface.co/saheedniyi/YarnGPT2) (Apache-2.0).

**How to use**
1. Click **Runtime → Change runtime type → T4 GPU** (free).
2. Click **Runtime → Run all**. (If you ran an older copy of this notebook, first click **Runtime → Restart session**.)
3. When it asks, upload `lines.json` (from the game's `voice/` folder).
4. Wait. It runs about 2–5 seconds per line on a T4, so the 62 lines take about 5 minutes.
5. A file called `nysc_voices.zip` downloads. Attach it in the chat with Claude.

Voices used: jude, chinenye, emma, umar, idera, joke, tayo, osagie, remi, zainab. Change the `voice` field in `lines.json` to swap anyone's voice.

In [ ]:
!git clone -q https://github.com/saheedniyi02/yarngpt.git
!pip -q install outetts uroman inflect gdown pydub
# outetts pins an old protobuf, which breaks Colab's preinstalled TensorFlow. YarnGPT does not use TensorFlow:
!pip -q uninstall -y tensorflow tensorflow-datasets tensorflow-metadata tf-keras ydf grain 2>/dev/null
!pip -q install -U "protobuf>=5.29,<7"
!wget -q https://huggingface.co/novateur/WavTokenizer-medium-speech-75token/resolve/main/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml
!gdown -q 1-ASeEkrn4HY49yZWHTASgfGFNXdVnLTt -O wavtokenizer_large_speech_320_24k.ckpt
print('install done (the red "dependency conflicts" text above is normal and safe to ignore)')

In [ ]:
import json, os, sys
from google.colab import files
print('Upload lines.json from the game folder (voice/lines.json)')
up = files.upload()
lines = json.load(open(list(up.keys())[0], encoding='utf-8'))
print(len(lines), 'lines')

In [ ]:
import os
os.environ['USE_TF'] = '0'; os.environ['TRANSFORMERS_NO_TF'] = '1'
import torch, torchaudio
from transformers import AutoModelForCausalLM
sys.path.append('yarngpt')
from yarngpt.audiotokenizer import AudioTokenizerV2
tok = AudioTokenizerV2('saheedniyi/YarnGPT2', '/content/wavtokenizer_large_speech_320_24k.ckpt', '/content/wavtokenizer_mediumdata_frame75_3s_nq1_code4096_dim512_kmeans200_attn.yaml')
model = AutoModelForCausalLM.from_pretrained('saheedniyi/YarnGPT2', torch_dtype='auto').to(tok.device)
print('model ready on', tok.device)

In [ ]:
import re, shutil, subprocess
from pydub import AudioSegment, effects
from pydub.silence import detect_leading_silence
shutil.rmtree('voice', ignore_errors=True); os.makedirs('voice', exist_ok=True)

def sentences(t):
    parts = [p.strip() for p in re.split(r'(?<=[.!?…])\s+', t) if p.strip()]
    out = []
    for p in parts:  # keep very short bits ("Again.") together with the next one so the model does not stumble
        if out and len(out[-1].split()) < 3: out[-1] = out[-1] + ' ' + p
        else: out.append(p)
    return out

def say_once(text, voice, temp):
    prompt = tok.create_prompt(text, lang='english', speaker_name=voice)
    ids = tok.tokenize_prompt(prompt)
    gen = model.generate(input_ids=ids, temperature=temp, repetition_penalty=1.1, max_length=4000)
    audio = tok.get_audio(tok.get_codes(gen))
    torchaudio.save('tmp.wav', audio, sample_rate=24000)
    seg = AudioSegment.from_wav('tmp.wav')
    a = detect_leading_silence(seg, silence_threshold=-45); b = detect_leading_silence(seg.reverse(), silence_threshold=-45)
    return seg[max(0, a - 60): len(seg) - max(0, b - 120)]

def say_sentence(text, voice):
    words = max(1, len(text.split()))
    best = None
    for temp in (0.1, 0.25, 0.4):  # retry if the length is clearly wrong (gibberish or cut off)
        seg = say_once(text, voice, temp)
        per_word = seg.duration_seconds / words
        if 0.28 <= per_word <= 0.95: return seg
        if best is None or abs(per_word - 0.45) < abs(best[1] - 0.45): best = (seg, per_word)
    print('   ⚠ kept best attempt for:', text[:50], round(best[1], 2), 's/word')
    return best[0]

done = []
for i, L in enumerate(lines):
    try:
        voice = L.get('voice', 'jude'); parts = sentences(L['text'])
        full = AudioSegment.silent(duration=80)
        for k, sct in enumerate(parts):
            full += say_sentence(sct, voice)
            if k < len(parts) - 1: full += AudioSegment.silent(duration=320)  # a real pause at the full stop
        full = effects.normalize(full) + AudioSegment.silent(duration=150)
        full.export('tmp_full.wav', format='wav')
        # slow down by 8% without changing the pitch
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', 'tmp_full.wav', '-filter:a', 'atempo=0.92', '-b:a', '64k', f"voice/{L['id']}.mp3"], check=True)
        done.append(L['id'])
        print(f"{i+1}/{len(lines)} ✓ [{voice}] {round(full.duration_seconds/0.92,1)}s  {L['text'][:60]}")
    except Exception as e:
        print(f"{i+1}/{len(lines)} ✗ {L['text'][:60]} → {e}")
json.dump(done, open('voice/manifest.json', 'w'))
print('done:', len(done), 'of', len(lines))

In [ ]:
!cd /content && zip -q -r nysc_voices.zip voice
files.download('nysc_voices.zip')